# Topic classifier

Roadmap step 3, part 2. Voteview's Clausen topics stop after the Congress beginning in 2013. This notebook tests whether a text classifier can label later roll calls well enough to extend the issue analysis.

- **Input:** each roll call's description text, which since 2001 is a short title such as "To amend the Immigration and Nationality Act...", plus vote type and chamber.
- **Model:** TF-IDF of words and word pairs with multinomial logistic regression. Its regularisation is chosen by cross-validation grouped by Congress.
- **Train:** coded roll calls from 2001 to 2012. **Test:** the Congress beginning in 2013, which the model never sees.
- **Baselines:** always guessing the most common topic, and a model that uses only vote type and chamber.

Only roll calls with description text can be classified: 65% in the House and nearly all in the Senate since 2001.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parents[1] / "data_loading"))
from explanation import ROLLCALL_KEYS, first_year, party_shares
from voteview import load_members, load_rollcalls, load_topics, load_votes

COLORS = {"D": "#2a78d6", "R": "#c93b3b"}
NEUTRAL = "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#b5b4ad",
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#e6e5df", "grid.linewidth": 0.8,
    "lines.linewidth": 2, "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "text.color": "#0b0b0b", "xtick.color": NEUTRAL, "ytick.color": NEUTRAL,
    "legend.frameon": False,
})
TOPICS = ["Government Management", "Foreign and Defense Policy", "Social Welfare", "Civil Liberties", "Agriculture", "Miscellaneous Policy"]
TRAIN = range(107, 113)
TEST = 113
PREDICT_FROM = 114
INCOMPLETE_CONGRESS = 119
OUTPUTS = Path.cwd().parent / "outputs"

topics = load_topics()
topics["vote_type"] = topics["vote_type"].fillna("none")
has_text = topics["text"].str.len() > 0
recent = topics[has_text & (topics["congress"] >= 107)].copy()
print("Share of roll calls with description text, by chamber, 2001 onward:")
print(topics[topics["congress"] >= 107].assign(has_text=has_text).groupby("chamber")["has_text"].mean().round(3).to_dict())
labelled = recent[recent["clausen"].notna() & (recent["congress"] <= TEST)]
train = labelled[labelled["congress"].isin(TRAIN)]
test = labelled[labelled["congress"] == TEST]
print(f"Training roll calls (2001 to 2012): {len(train):,}; test roll calls (2013 to 2014): {len(test):,}")
print("Example descriptions:")
for row in train.sample(5, random_state=0).itertuples():
    print(f"  [{row.clausen}] {row.text[:110]}")


Share of roll calls with description text, by chamber, 2001 onward:
{'House': 0.647, 'Senate': 0.997}
Training roll calls (2001 to 2012): 9,325; test roll calls (2013 to 2014): 1,353
Example descriptions:
  [Government Management] To provide a reduction in State estate tax revenues in proportion to the reduction in Federal estate tax reven
  [Foreign and Defense Policy] To amend the Immigration and Nationality Act to toll, during active-duty service abroad in the Armed Forces, t
  [Government Management] United States-Morocco Free Trade Agreement Implementation Act
  [Foreign and Defense Policy] Recognizing the 25th anniversary of the adoption of the Constitution of the Republic of the Marshall Islands
  [Social Welfare] To increase appropriations for the Low-Income Home Energy Assistance Program by $1,276,000,000, with an across


9,325 roll calls for training and 1,353 for testing. The text is short and specific, which suits a bag-of-words model.

## Training and testing the classifier

In [2]:
def text_model(C):
    features = ColumnTransformer([
        ("text", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode"), "text"),
        ("type", OneHotEncoder(handle_unknown="ignore"), ["vote_type", "chamber"]),
    ])
    return make_pipeline(features, LogisticRegression(C=C, max_iter=3000))


def type_model():
    features = ColumnTransformer([("type", OneHotEncoder(handle_unknown="ignore"), ["vote_type", "chamber"])])
    return make_pipeline(features, LogisticRegression(max_iter=3000))


columns = ["text", "vote_type", "chamber"]
search = {}
for C in [1, 4, 16, 64]:
    scores = cross_val_score(text_model(C), train[columns], train["clausen"], groups=train["congress"],
                             cv=GroupKFold(n_splits=3), scoring="f1_macro")
    search[C] = scores.mean()
best_C = max(search, key=search.get)
print("Macro F1 by regularisation strength C, cross-validated by Congress on the training years:", {k: round(v, 3) for k, v in search.items()})

model = text_model(best_C).fit(train[columns], train["clausen"])
majority = train["clausen"].value_counts().idxmax()
baseline = type_model().fit(train[columns], train["clausen"])
results = pd.DataFrame({
    "accuracy": [accuracy_score(test["clausen"], [majority] * len(test)),
                 accuracy_score(test["clausen"], baseline.predict(test[columns])),
                 accuracy_score(test["clausen"], model.predict(test[columns]))],
    "macro_f1": [f1_score(test["clausen"], [majority] * len(test), average="macro"),
                 f1_score(test["clausen"], baseline.predict(test[columns]), average="macro"),
                 f1_score(test["clausen"], model.predict(test[columns]), average="macro")],
}, index=[f"Most common topic ({majority})", "Vote type and chamber only", f"Text model (C={best_C})"])
print(f"Held-out test: Congress {TEST} (2013 to 2014)")
display(results.round(3))
print(classification_report(test["clausen"], model.predict(test[columns]), digits=3))

earlier = text_model(best_C).fit(train[train["congress"] < 112][columns], train[train["congress"] < 112]["clausen"])
later = train[train["congress"] == 112]
print(f"Stability check, trained on 2001 to 2010 and tested on 2011 to 2012: accuracy {accuracy_score(later['clausen'], earlier.predict(later[columns])):.3f}, "
      f"macro F1 {f1_score(later['clausen'], earlier.predict(later[columns]), average='macro'):.3f}")
confusion = pd.crosstab(test["clausen"], model.predict(test[columns]), rownames=["actual"], colnames=["predicted"], normalize="index")
display(confusion.reindex(index=TOPICS, columns=TOPICS).round(2))


Macro F1 by regularisation strength C, cross-validated by Congress on the training years: {1: np.float64(0.517), 4: np.float64(0.57), 16: np.float64(0.568), 64: np.float64(0.573)}


Held-out test: Congress 113 (2013 to 2014)


,accuracy,macro_f1
Most common topic (Government Management),0.704,0.138
Vote type and chamber only,0.704,0.138
Text model (C=64),0.822,0.614


                            precision    recall  f1-score   support

               Agriculture      0.583     0.519     0.549        27
           Civil Liberties      0.467     0.233     0.311        30
Foreign and Defense Policy      0.717     0.653     0.683       167
     Government Management      0.870     0.923     0.896       952
      Miscellaneous Policy      0.706     0.545     0.615        22
            Social Welfare      0.674     0.587     0.628       155

                  accuracy                          0.822      1353
                 macro avg      0.670     0.577     0.614      1353
              weighted avg      0.812     0.822     0.815      1353



Stability check, trained on 2001 to 2010 and tested on 2011 to 2012: accuracy 0.751, macro F1 0.550


predicted,Government Management,Foreign and Defense Policy,Social Welfare,Civil Liberties,Agriculture,Miscellaneous Policy
actual,,,,,,
Government Management,0.92,0.03,0.03,0.01,0.00,0.00
Foreign and Defense Policy,0.30,0.65,0.03,0.01,0.00,0.01
Social Welfare,0.32,0.06,0.59,0.01,0.03,0.00
Civil Liberties,0.43,0.13,0.13,0.23,0.07,0.00
Agriculture,0.41,0.04,0.04,0.00,0.52,0.00
Miscellaneous Policy,0.32,0.00,0.05,0.00,0.09,0.55


**What this shows**

- **The text model clearly beats both baselines.** On the 2013 to 2014 test Congress, accuracy is 82.2% against 70.4%, and macro F1 0.61 against 0.14. Vote type and chamber alone add nothing over guessing Government Management.
- **Performance is uneven by topic:**
  - Government Management, the large majority class, is labelled well (F1 0.90);
  - Foreign and Defense Policy (0.68) and Social Welfare (0.63) are moderate;
  - Civil Liberties is poor (F1 0.31, recall 0.23), with many of its votes labelled Government Management.
- **Errors mostly fall into Government Management,** as the confusion table shows, so smaller topics are under-counted.
- **Model selection:** cross-validation chose C = 64, the largest value tried, but scores barely change between 4 and 64 (macro F1 0.568 to 0.573). The folds group by Congress but don't move forward in time.
- **Stability check:** trained on 2001 to 2010 and tested on 2011 to 2012, accuracy is 75.1% and macro F1 0.55. That check reuses the C chosen with 2011 to 2012 inside cross-validation, so it is slightly optimistic. Performance varies between Congresses, and 2013 to 2014 may be a favourable test.

## How much would errors distort issue-level measures?

In [3]:
members = load_members()
members = members[~members["is_delegate"]]
votes = load_votes()
votes = votes[votes["congress"] >= 107].merge(members[["congress", "chamber", "icpsr", "party"]], on=["congress", "chamber", "icpsr"])
rollcalls = load_rollcalls().set_index(ROLLCALL_KEYS)
shares = party_shares(votes).join(rollcalls[["is_lopsided"]])
del votes
shares = shares[~shares["is_lopsided"]].reset_index()
shares["opposed"] = (shares["D"] - 0.5) * (shares["R"] - 0.5) < 0

test_frame = test.assign(predicted=model.predict(test[columns])).merge(shares, on=ROLLCALL_KEYS)
compare = pd.DataFrame({
    "actual_share": test_frame["clausen"].value_counts(normalize=True),
    "predicted_share": test_frame["predicted"].value_counts(normalize=True),
    "opposed_by_actual_topic": test_frame.groupby("clausen")["opposed"].mean(),
    "opposed_by_predicted_topic": test_frame.groupby("predicted")["opposed"].mean(),
}).reindex(TOPICS)
compare["error_points"] = (compare["opposed_by_predicted_topic"] - compare["opposed_by_actual_topic"]) * 100
print(f"How prediction errors would change topic-level measures, contested roll calls in Congress {TEST}, both chambers")
display(compare.round(3))
by_chamber = {}
for chamber, part in test_frame.groupby("chamber"):
    actual = part.groupby("clausen")["opposed"].mean()
    predicted = part.groupby("predicted")["opposed"].mean()
    by_chamber[(chamber, "error_points")] = (predicted - actual) * 100
    by_chamber[(chamber, "rollcalls")] = part["clausen"].value_counts()
print("Errors by chamber (points), with the number of test roll calls by actual topic")
display(pd.DataFrame(by_chamber).reindex(TOPICS).round(1))


How prediction errors would change topic-level measures, contested roll calls in Congress 113, both chambers


,actual_share,predicted_share,opposed_by_actual_topic,opposed_by_predicted_topic,error_points
Government Management,0.704,0.755,0.857,0.855,-0.262
Foreign and Defense Policy,0.100,0.087,0.611,0.691,8.038
Social Welfare,0.131,0.112,0.894,0.851,-4.313
Civil Liberties,0.021,0.009,0.826,0.700,-12.609
Agriculture,0.024,0.022,0.769,0.750,-1.923
Miscellaneous Policy,0.020,0.016,0.909,0.824,-8.556


Errors by chamber (points), with the number of test roll calls by actual topic


House                 Senate          
                           error_points rollcalls error_points rollcalls
Government Management              -0.0       348         -0.6       415
Foreign and Defense Policy          4.0        59         12.9        49
Social Welfare                     -3.3        92         -5.3        50
Civil Liberties                   -20.8         8         -8.6        15
Agriculture                         0.0        15          0.0        11
Miscellaneous Policy                0.0        12        -22.9        10

**What this shows**

The real question is how far errors move the measure we care about: the share of roll calls with party majorities opposed, by topic.

On the 2013 to 2014 test Congress, with both chambers together:

- Government Management is off by -0.3 points, Foreign and Defense Policy by +8.0, and Social Welfare by -4.3.
- Small topics are worse, as the table shows.

By chamber:

- **Senate Foreign and Defense Policy** is overstated by 12.9 points (from 49 test roll calls); in the House, by 4.0.
- Civil Liberties is off by -20.8 points in the House, but on only 8 roll calls.

These errors come from a single Congress, so they are a rough guide rather than a bound.

Predicted topic shares are also distorted toward Government Management (75.5% predicted against 70.4% actual), so the topic mix after 2014 shouldn't be compared directly with coded years.

## Topics after 2014

In [4]:
future = recent[recent["congress"] >= PREDICT_FROM].copy()
future["predicted"] = model.predict(future[columns])
future["confidence"] = model.predict_proba(future[columns]).max(axis=1)
OUTPUTS.mkdir(exist_ok=True)
future[ROLLCALL_KEYS + ["predicted", "confidence"]].to_parquet(OUTPUTS / "predicted_topics.parquet", index=False)
all_future = topics[topics["congress"] >= PREDICT_FROM]
print(f"Predicted topics for {len(future):,} of {len(all_future):,} roll calls from 2015 on; the rest have no description text.")
print("Median confidence (highest class probability):", round(future["confidence"].median(), 3))

combined = pd.concat([
    labelled.merge(shares, on=ROLLCALL_KEYS).assign(topic=lambda f: f["clausen"], source="coded"),
    future.merge(shares, on=ROLLCALL_KEYS).assign(topic=lambda f: f["predicted"], source="predicted"),
])
combined = combined[combined["congress"] < INCOMPLETE_CONGRESS]
combined["period"] = pd.cut(first_year(combined["congress"]), [2000, 2008, 2014, 2024], labels=["2001-2007 (coded)", "2009-2013 (coded)", "2015-2023 (predicted)"])
table = combined.pivot_table(index=["chamber", "period"], columns="topic", values="opposed", aggfunc="mean", observed=True)
print("Share of contested roll calls with party majorities opposed, by topic")
display(table.reindex(columns=TOPICS).round(3))
print("Number of contested roll calls behind each share")
display(combined.pivot_table(index=["chamber", "period"], columns="topic", values="opposed", aggfunc="size", observed=True).reindex(columns=TOPICS).fillna(0).astype(int))
sizes = combined.groupby(["chamber", "period"], observed=True)["topic"].value_counts(normalize=True).unstack()
print("Topic mix")
display(sizes.reindex(columns=TOPICS).round(3))


Predicted topics for 8,552 of 10,762 roll calls from 2015 on; the rest have no description text.
Median confidence (highest class probability): 0.941
Share of contested roll calls with party majorities opposed, by topic


topic                          Government Management  \
chamber period                                         
House   2001-2007 (coded)                      0.760   
        2009-2013 (coded)                      0.841   
        2015-2023 (predicted)                  0.785   
Senate  2001-2007 (coded)                      0.716   
        2009-2013 (coded)                      0.797   
        2015-2023 (predicted)                  0.773   

topic                          Foreign and Defense Policy  Social Welfare  \
chamber period                                                              
House   2001-2007 (coded)                           0.637           0.802   
        2009-2013 (coded)                           0.690           0.846   
        2015-2023 (predicted)                       0.690           0.838   
Senate  2001-2007 (coded)                           0.734           0.807   
        2009-2013 (coded)                           0.695           0.909   
        2015-2023 (predicted)                       0.590           0.871   

topic                          Civil Liberties  Agriculture  \
chamber period                                                
House   2001-2007 (coded)                0.760        0.628   
        2009-2013 (coded)                0.838        0.905   
        2015-2023 (predicted)            0.799        0.889   
Senate  2001-2007 (coded)                0.770        0.632   
        2009-2013 (coded)                0.531        0.662   
        2015-2023 (predicted)            0.686        0.200   

topic                          Miscellaneous Policy  
chamber period                                       
House   2001-2007 (coded)                     0.919  
        2009-2013 (coded)                     0.975  
        2015-2023 (predicted)                 0.949  
Senate  2001-2007 (coded)                     0.560  
        2009-2013 (coded)                     0.724  
        2015-2023 (predicted)                 0.389

Number of contested roll calls behind each share


topic                          Government Management  \
chamber period                                         
House   2001-2007 (coded)                       1151   
        2009-2013 (coded)                       1173   
        2015-2023 (predicted)                   1729   
Senate  2001-2007 (coded)                        986   
        2009-2013 (coded)                       1057   
        2015-2023 (predicted)                   2494   

topic                          Foreign and Defense Policy  Social Welfare  \
chamber period                                                              
House   2001-2007 (coded)                             444             303   
        2009-2013 (coded)                             226             221   
        2015-2023 (predicted)                         484             376   
Senate  2001-2007 (coded)                             433             394   
        2009-2013 (coded)                             167             208   
        2015-2023 (predicted)                         366             232   

topic                          Civil Liberties  Agriculture  \
chamber period                                                
House   2001-2007 (coded)                  171           43   
        2009-2013 (coded)                   37           42   
        2015-2023 (predicted)              134           18   
Senate  2001-2007 (coded)                  139           68   
        2009-2013 (coded)                   49           65   
        2015-2023 (predicted)               35           10   

topic                          Miscellaneous Policy  
chamber period                                       
House   2001-2007 (coded)                        86  
        2009-2013 (coded)                        80  
        2015-2023 (predicted)                    79  
Senate  2001-2007 (coded)                        25  
        2009-2013 (coded)                        29  
        2015-2023 (predicted)                    18

Topic mix


topic                          Government Management  \
chamber period                                         
House   2001-2007 (coded)                      0.524   
        2009-2013 (coded)                      0.659   
        2015-2023 (predicted)                  0.613   
Senate  2001-2007 (coded)                      0.482   
        2009-2013 (coded)                      0.671   
        2015-2023 (predicted)                  0.790   

topic                          Foreign and Defense Policy  Social Welfare  \
chamber period                                                              
House   2001-2007 (coded)                           0.202           0.138   
        2009-2013 (coded)                           0.127           0.124   
        2015-2023 (predicted)                       0.172           0.133   
Senate  2001-2007 (coded)                           0.212           0.193   
        2009-2013 (coded)                           0.106           0.132   
        2015-2023 (predicted)                       0.116           0.074   

topic                          Civil Liberties  Agriculture  \
chamber period                                                
House   2001-2007 (coded)                0.078        0.020   
        2009-2013 (coded)                0.021        0.024   
        2015-2023 (predicted)            0.048        0.006   
Senate  2001-2007 (coded)                0.068        0.033   
        2009-2013 (coded)                0.031        0.041   
        2015-2023 (predicted)            0.011        0.003   

topic                          Miscellaneous Policy  
chamber period                                       
House   2001-2007 (coded)                     0.039  
        2009-2013 (coded)                     0.045  
        2015-2023 (predicted)                 0.028  
Senate  2001-2007 (coded)                     0.012  
        2009-2013 (coded)                     0.018  
        2015-2023 (predicted)                 0.006

**What this shows**

The classifier labels 8,552 of 10,762 roll calls from 2015 on; the rest have no description text. Its median confidence is 0.94. **These are predictions, not codes.** Given the test results, only the three large topics are reported.

From 2015 to 2023, the predicted share of contested roll calls with party majorities opposed, against 2009 to 2013 (coded):

- **Government Management:** 79% in the House and 77% in the Senate, against 84% and 80%.
- **Social Welfare:** 84% and 87%, against 85% and 91%.
- **Foreign and Defense Policy:** 69% in the House, unchanged; **59% in the Senate, against 70%.**

How to read these:

- **Division on Government Management and Social Welfare stayed high.** The differences are within the test errors.
- **Senate Foreign and Defense Policy may have become less divided.** Its 11-point fall is larger than the test error, and on the test Congress the predicted labels overstated Senate opposition on this topic by about 13 points, so the true fall could be larger. Treat this as a possible change to check, not a finding.
- **Small topics after 2014 aren't usable.** For example, Senate Agriculture has only 8 predicted contested roll calls.
- House roll calls without text (about a third) are left out, and they may not be a random subset.

Predictions are saved to `outputs/predicted_topics.parquet`, which isn't committed.

## Summary

- **A simple text classifier labels recent roll calls' topics with 82% accuracy**, against 70% for always guessing the most common topic. It does well on large topics and poorly on small ones, especially Civil Liberties.
- **Topic-level measures from predicted labels are within a few points for Government Management and Social Welfare.** Errors are larger for Senate Foreign and Defense Policy and for small topics.
- **From 2015 to 2023, division on Government Management and Social Welfare stayed high.** Senate Foreign and Defense Policy may have become less divided, which needs checking with coded data.
- **Possible improvements:**
  - add CRS policy areas and subjects as features where available;
  - train on older long descriptions too;
  - validate with forward-moving splits over several Congresses.